# Use Case 5: Reranking Search Results (RAG retrieval)

**The real-world problem:** "RAG" (Retrieval-Augmented Generation) means: before answering a question, first go *find* the relevant piece of text, then use that as context. The tricky part is step 1 — finding it.

Fast search methods (keyword search, or the vector/embedding search you may have heard of) are quick but kind of dumb — they're good at finding candidates that are *roughly* about the right topic, but bad at telling which one **actually** answers the question. That's what **reranking** fixes: take a short list of maybe-relevant candidates, and use something smarter to carefully re-sort them by true relevance.

**Good news, again:** no new TypeSafe concept. Reranking is just `Noul` (from notebook 3) — asked once per candidate — plus one line of plain Python (`sorted()`) to reorder them by the score.

### 📚 Analogy

A visitor asks a librarian: "How long do I have to return something for a refund?"

A junior librarian does a **quick catalog search** and grabs 5 books that mention words like "return" or "refund" — fast, but not very smart. Some of those books might not actually answer the question.

A senior librarian then goes through **each book, one at a time**, asking the exact same yes/no question every time: *"Does this book actually answer what they asked?"* — and writes a confidence number (0 to 1) on a sticky note for each one.

Once every book has a sticky note, the librarian **sorts the stack** by that number, highest first, and hands over the top one.

### 🗺️ The 4 things we'll actually do in this notebook

1. Check **one** candidate snippet, exactly like notebook 3 did with one message.
2. Do that **same check 5 times**, once per candidate — this is where we learn Python's "for loop", so we don't have to copy-paste the same code 5 times.
3. **Sort** the 5 results by their score, highest first.
4. Look at the **winner** and decide if it's actually good enough.

## Step 1 — Install the SDK

In [1]:
%pip install typesafe-sdk

Note: you may need to restart the kernel to use updated packages.


## Step 2 — Add your API key

In [ ]:
import os

# 👉 PASTE YOUR TYPESAFE API KEY BELOW, between the quotes
os.environ["TYPESAFE_API_KEY"] = "PASTE_YOUR_API_KEY_HERE"

print("Key saved for this session.")

## Step 3 — Create the client

We import `NoulCriteria` this time too — a small helper that lets us clarify exactly what counts as "yes" vs "no", which helps when candidates are only *kind of* related (more on this in Step 5).

In [3]:
from typesafe_sdk import Noul, NoulCriteria, TypeSafeClient

client = TypeSafeClient()
print("Client is ready.")

Client is ready.


## Step 4 — The shortlist (pretend a fast search already found these)

In a real app, this shortlist would come from a fast keyword or vector search over thousands of documents. To keep this notebook simple, we'll skip that step and just pretend we already have 5 candidate FAQ snippets to check.

In [4]:
query = "How long do I have to return an item for a refund?"

candidates = [
    "Our support team is available 24/7 via live chat and email.",
    "Items can be returned within 30 days of purchase for a full refund, as long as they're unused and in original packaging.",
    "Refunds are processed within 5-7 business days after we receive your returned item.",
    "We offer free shipping on orders over $50.",
    "Exchanges for a different size are accepted within 45 days of purchase.",
]

Notice these are deliberately tricky: none of them is *obviously* wrong, but only one of them actually answers the question. The others are near misses:
- Snippet 3 is about refund *processing time*, not the *return window* — a different question that happens to share the word "refund".
- Snippet 5 is about *exchanges*, not *refunds* — a related but different policy.
- Snippets 1 and 4 are just unrelated.

## Step 5 — Define the question (once), reused for every candidate

Because some candidates are near misses rather than obviously wrong, we add `criteria` this time — like notebook 3 mentioned, this is exactly when it's worth spelling out the true/false boundary explicitly instead of leaving it implicit.

In [5]:
def relevance_question(user_query):
    return Noul(
        instructions=f'Does this passage directly answer the question: "{user_query}"?',
        criteria=NoulCriteria(
            true="The passage states the specific information the question is asking for.",
            false="The passage is on a related topic but does not give that specific answer, or is unrelated.",
        ),
    )

## Step 6 — Score every candidate against the query

We loop over the 5 candidates, and for each one send **both the query and that one candidate** as the `state` (this is the same pattern as always — just this time `state` holds two pieces of text instead of one). We collect each `noul` score as we go.

In [7]:
scored_candidates = []

for text in candidates:
    print(f"Scoring candidate: {text}")
    print(f"Query: {query}")
    response = client.system_one(
        state={"query": query, "candidate": text},
        questions={"is_relevant": relevance_question(query)},
    )
    score = response.nouls["is_relevant"].noul
    scored_candidates.append({"text": text, "score": score})
    print(f"{score:.2f}  {text}")

Scoring candidate: Our support team is available 24/7 via live chat and email.
Query: How long do I have to return an item for a refund?
0.01  Our support team is available 24/7 via live chat and email.
Scoring candidate: Items can be returned within 30 days of purchase for a full refund, as long as they're unused and in original packaging.
Query: How long do I have to return an item for a refund?
0.98  Items can be returned within 30 days of purchase for a full refund, as long as they're unused and in original packaging.
Scoring candidate: Refunds are processed within 5-7 business days after we receive your returned item.
Query: How long do I have to return an item for a refund?
0.04  Refunds are processed within 5-7 business days after we receive your returned item.
Scoring candidate: We offer free shipping on orders over $50.
Query: How long do I have to return an item for a refund?
0.01  We offer free shipping on orders over $50.
Scoring candidate: Exchanges for a different size ar

## Step 7 — Rerank: sort by score, highest first

This is the entire "reranking" step. No AI here — just Python's built-in `sorted()`, using the scores we just collected.

In [ ]:
reranked = sorted(scored_candidates, key=lambda c: c["score"], reverse=True)

print("Query:", query)
print()
print("Ranked results (best first):")
for rank, c in enumerate(reranked, start=1):
    print(f"{rank}. [{c['score']:.2f}] {c['text']}")

The 30-day return policy snippet should land in 1st place with a high score, and the other 4 should trail behind it — including the two tricky near-misses (refund *processing time*, and *exchanges*), which a plain keyword search would likely have ranked just as high as the real answer, since they all share words like "refund" or "days".

## Step 8 — Bonus: what if nothing actually answers the question?

Sorting always produces a "winner" — even if every candidate is actually irrelevant, one of them still ends up on top. So it's worth checking the **top score itself**, not just its rank, exactly like the threshold check from notebook 4.

In [ ]:
top_result = reranked[0]

if top_result["score"] > 0.5:
    print("✅ Best match found:", top_result["text"])
else:
    print("⚠️ No candidate really answers this question (top score was only", round(top_result["score"], 2), ")")

## Step 9 — Try it yourself

Change `query_2` below to something **none** of our 5 candidates actually answer (e.g. "Do you ship internationally?") and re-run. Watch every score come back low, and the bonus check in Step 8's style correctly say "no match" instead of confidently pointing at the wrong snippet.

In [ ]:
# ✏️ Edit this, then re-run this cell (Shift+Enter)
query_2 = "Do you ship internationally?"

scored_2 = []
for text in candidates:
    response = client.system_one(
        state={"query": query_2, "candidate": text},
        questions={"is_relevant": relevance_question(query_2)},
    )
    scored_2.append({"text": text, "score": response.nouls["is_relevant"].noul})

reranked_2 = sorted(scored_2, key=lambda c: c["score"], reverse=True)

print("Query:", query_2)
for rank, c in enumerate(reranked_2, start=1):
    print(f"{rank}. [{c['score']:.2f}] {c['text']}")

top_2 = reranked_2[0]
if top_2["score"] > 0.5:
    print("\n✅ Best match found:", top_2["text"])
else:
    print("\n⚠️ No candidate really answers this question (top score was only", round(top_2["score"], 2), ")")

## Step 10 — Clean up

In [ ]:
client.close()
print("Closed.")

## Recap — what did we just learn?

1. **Reranking** = re-sorting a shortlist of candidates by *true* relevance to a query, after a faster (but dumber) search already narrowed things down.
2. It's built from a single `Noul` question — "does this candidate answer the query?" — asked once per candidate, with `state` holding **both** the query and that one candidate.
3. `NoulCriteria(true=..., false=...)` is worth adding whenever candidates can be near-misses (related topic, wrong specifics) rather than obviously right or wrong.
4. The actual "reranking" is just `sorted(..., key=..., reverse=True)` — one line of ordinary Python, no AI involved in that step.
5. Always sanity-check the **top score itself**, not just its rank — sorting always produces a #1, even when nothing genuinely answers the query.

*(For reference: TypeSafe's own reranking cookbook does the exact same thing at bigger scale — thousands of documents, a real fast-search first pass (BM25), and many candidate checks fired off at once instead of one at a time in a loop, for speed. In one published test on legal documents, adding this reranking step improved top-1 search accuracy from 5% to 18%. The core idea — one `Noul` per candidate, then sort — is identical to what you just built.)*

👉 Tell me which use case to build next whenever you're ready.